In [91]:
import warnings
warnings.filterwarnings('ignore')

In [92]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

from zipfile import ZipFile
from urllib.request import urlopen
from io import BytesIO
import unicodedata

import joblib

In [93]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [94]:
# Source: Wikipedia, FEC
# Links:
# https://en.wikipedia.org/wiki/2024_United_States_presidential_election#Electoral_results
# https://www.fec.gov/resources/cms-content/documents/federalelections2020.pdf
# https://www.fec.gov/resources/cms-content/documents/federalelections2016.pdf#page=10
# https://www.fec.gov/resources/cms-content/documents/federalelections2012.pdf#page=11
dem_2pv_24 = 75017613 / (75017613 + 77302580) * 100
dem_2pv_20 = 81283501 / (81283501 + 74223975) * 100
dem_2pv_16 = 65853514 / (65853514 + 62984828) * 100
dem_2pv_12 = 65915795 / (65915795 + 60933504) * 100
dem_2pv_08 = 69498516 / (69498516 + 59948323) * 100

pop_vote = {
    '2008': dem_2pv_08,
    '2012': dem_2pv_12,
    '2016': dem_2pv_16,
    '2020': dem_2pv_20,
    '2024': dem_2pv_24
}

# dem_2pv_08, dem_2pv_12, dem_2pv_16, dem_2pv_20, dem_2pv_24
pop_vote

{'2008': 53.68884751214358,
 '2012': 51.96386225200976,
 '2016': 51.113288930712876,
 '2020': 52.26983492420647,
 '2024': 49.24994613156773}

In [95]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTZTum7NEaS6hvniyNJbDTWLDY0rATtgR5yXV2EUbtt6ubRPqv1sVH3PdJM3rgAzS_ak_XSgxLtaq3-/pub?output=csv'
candinfo = pd.read_csv(url)
candinfo.to_csv('../../2026_data/2026_senate_candidateinfo.csv')
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any
0,Alabama,Everett Weiss,Barry Moore,False,False
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True
3,Colorado,John Hickenlooper,Mark Baisley,True,False
4,Delaware,Chris Coons,Michael Katz,True,False


In [96]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_cand'] = candinfo[f'{party}_cand'].fillna('TBD').astype(str)
    candinfo[f'{party}_inc_any'] = candinfo[f'{party}_inc_any'].astype(bool)

In [97]:
dem_only_candinfo = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_only_candinfo = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]

In [98]:
candinfo['state_po'] = candinfo['state'].map(us_state_to_abbrev)
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po
0,Alabama,Everett Weiss,Barry Moore,False,False,AL
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO
4,Delaware,Chris Coons,Michael Katz,True,False,DE


In [99]:
candinfo.shape

(35, 6)

In [100]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [101]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

In [102]:
webl = pd.read_table('../../2026_data/fec/webl26.txt', sep="|", names=fec_webl_colnames)
cn = pd.read_table('../../2026_data/fec/cn.txt', sep="|", names=fec_cn_colnames)
fec = pd.merge(left=webl, right=cn, on='CAND_ID', how='left')
fec = fec[[col for col in fec.columns.values if ('_y' not in col)]]
fec.columns = fec.columns.str.strip('_x')
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",C,1,DEM,152304.86,0.00,232791.29,0.0,83969.49,...,0.00,2026.0,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501.0
1,H2AK01083,"BEGICH, NICHOLAS III",I,2,REP,5470446.24,1601616.78,2455618.51,0.0,104330.06,...,4382.86,2026.0,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567.0
2,H6AK01092,"HILL, BILL",C,3,IND,1336291.63,7400.00,820405.67,0.0,0.00,...,0.00,2026.0,H,C,C00935437,PO BOX 220703,NaN,ANCHORAGE,AK,99522.0
3,H6AL01094,"JONES, CLYDE W MR. JR",O,1,DEM,85967.63,0.00,57937.52,0.0,0.00,...,0.00,2026.0,H,C,C00920918,11637 WENTWOOD CT,NaN,DAPHNE,AL,36526.0
4,H0AL01055,"CARL, JERRY LEE, JR",O,2,REP,1162582.80,183453.86,865256.75,0.0,69290.42,...,1000.00,2026.0,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685.0


In [103]:
fec = fec[fec['CAND_OFFICE'] == 'S']
fec['CAND_OFFICE_ST'] = fec['CAND_OFFICE_ST'].astype(str)
fec['CAND_NAME'] = fec['CAND_NAME'].astype(str)
fec['CAND_PTY_AFFILIATION'] = fec['CAND_PTY_AFFILIATION'].replace({'DFL': 'DEM'})

In [104]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype=object)

In [105]:
fec.shape

(336, 39)

In [106]:
def get_fuzzymatch_cand(state_po, party, candidate):
    error_return = 'no_match'
    if candidate == 'TBD':
        return error_return
    
    df = fec[(fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'NON', 'OTH']))]

    # df['cand_name_lst'] = df['CAND_NAME'].str.split(',')
    # df['first_name_cand'] = df['cand_name_lst'].map(lambda x: x[1])
    # df['last_name_cand'] = df['cand_name_lst'].map(lambda x: x[0])
    # df['cand_name_ordered'] = df['first_name_cand'] + ' ' + df['last_name_cand']

    if df.shape[0] == 0:
        return error_return

    if '/' in candidate:
        cands = candidate.split(separator='/')
        matches, contribs = [], []
        for c in cands:
            fuzzymatch = process.extractOne(c, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=60)
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch]['TTL_INDIV_CONTRIB'].values[0]
            if fuzzymatch is not None:
                matches.append(fuzzymatch[0])
                contribs.append(indiv_contrib)
            else:
                matches.append(error_return)
                contribs.append(0)
        return matches

    else:
        fuzzymatch = process.extractOne(candidate, df['CAND_NAME'].values, scorer=fuzz.token_sort_ratio, score_cutoff=10)
        if fuzzymatch is not None:
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch[0]]['TTL_INDIV_CONTRIB'].values[0]
        else:
            indiv_contrib = 0
        return error_return if fuzzymatch is None else fuzzymatch[0]

In [107]:
candinfo['fec_dem_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], 'DEM', x['dem_cand']), axis=1)
candinfo['fec_rep_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], 'REP', x['rep_cand']), axis=1)
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"SWEETSER, KYLE","MOORE, FELIX BARRY"
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J"
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS"
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK"
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR."


In [108]:
dem_corrections = {
    ('Todd Achilles', 'ID'):        'ACHILLES, TODD BAKER',
    ('Everett Weiss', 'AL'):        'WESS, EVERETT W',
    ('Peggy Flanagan', 'MN'):       'FLANAGAN, MARGARET',
    ('Troy Jackson', 'ME'):         'no_match',
    ('Marquita Bradshaw', 'TN'):    'BRADSHAW, MARQUITA',
}

rep_corrections = {
    ('Mike Collins', 'GA'):         'COLLINS, MICHAEL A JR',
    ('Jim Risch', 'ID'):            'RISCH, JAMES E MR.',
    ('Ken Paxton', 'TX'):           'PAXTON, WARREN KENNETH JR.',
    ('Michael Whatley', 'NC'):      'WHATLEY, MICHAEL',
}

In [109]:
for (cand, state) in dem_corrections.keys():
    mask = (
        (candinfo['state_po'] == state) &
        (candinfo['dem_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_dem_fuzzymatch'] = dem_corrections[(cand, state)]

for (cand, state) in rep_corrections.keys():
    mask = (
        (candinfo['state_po'] == state) &
        (candinfo['rep_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_rep_fuzzymatch'] = rep_corrections[(cand, state)]

In [110]:
candinfo

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY"
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J"
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS"
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK"
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR."
5,Florida,Angie Nixon,Ashley Moody,False,True,FL,"NIXON, ANGELA","MOODY, ASHLEY"
6,Georgia,Jon Ossoff,Mike Collins,True,False,GA,"OSSOFF, T. JONATHAN","COLLINS, MICHAEL A JR"
7,Idaho,Todd Achilles,Jim Risch,False,True,ID,"ACHILLES, TODD BAKER","RISCH, JAMES E MR."
8,Illinois,Juliana Stratton,Don Tracy,False,False,IL,"STRATTON, JULIANA","TRACY, DON"
9,Iowa,Josh Turek,Ashley Hinson,False,False,IA,"TUREK, JOSHUA","ARENHOLZ, ASHLEY HINSON"


In [111]:
def get_indiv_contribs(state_po, party, candidate):
    if candidate == 'TBD':
        return 0
    
    df = fec[
        (fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'OTH', 'NON'])) &
        (fec['CAND_NAME'] == candidate)
    ]

    if df.shape[0] == 0:
        return 0

    return df['TTL_INDIV_CONTRIB'].values[0]

candinfo['dem_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], 'DEM', x['fec_dem_fuzzymatch']), axis=1)
candinfo['rep_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], 'REP', x['fec_rep_fuzzymatch']), axis=1)
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27


In [112]:
candinfo['2p_funds'] = candinfo['dem_funds'] + candinfo['rep_funds']
candinfo['dem_funds_2p_pct'] = candinfo['dem_funds'] / candinfo['2p_funds'] * 100
candinfo['rep_funds_2p_pct'] = candinfo['rep_funds'] / candinfo['2p_funds'] * 100
for party in ['dem', 'rep']:
    candinfo[f'{party}_funds_2p_pct'] = candinfo[f'{party}_funds_2p_pct'].fillna(0) 
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874


In [113]:
pres = pd.read_csv('../../data/pres_results/1976-2024-president.csv')
pres = pres[(pres['year'] >= 2020) & (~pres['candidate'].isin(['PAIJ BORING AND JAIMEY RUTSCHMAN', 'SETHATINA NEWMAN']))]
pres_pivot = pd.pivot_table(pres, values='candidatevotes', index=['year', 'state'], columns='party_simplified',
                           aggfunc='first')[['DEMOCRAT', 'REPUBLICAN']]
pres_pivot = pres_pivot.reset_index()
pres_pivot = pres_pivot[(~pres_pivot['DEMOCRAT'].isna()) | (~pres_pivot['REPUBLICAN'].isna())]
pres_pivot = pres_pivot.rename({
    'DEMOCRAT': 'dem',
    'REPUBLICAN': 'rep'
}, axis=1)
pres_pivot['tot'] = pres_pivot['dem'] + pres_pivot['rep']
for party in ['dem', 'rep']:
    pres_pivot[f'{party}_2p_pct'] = pres_pivot[party] / pres_pivot['tot'] * 100
pres_pivot['state_po'] = pres_pivot['state'].map(lambda x: x.title()).map(us_state_to_abbrev)
pres_pivot = pres_pivot[~pres_pivot['state_po'].isna()]
pres_pivot['year'] = pres_pivot['year'].astype(int)
pres_pivot['lean'] = pres_pivot['dem_2p_pct'] - pres_pivot['year'].astype(str).map(pop_vote)
pres_pivot.head(10)

party_simplified,year,state,dem,rep,tot,dem_2p_pct,rep_2p_pct,state_po,lean
0,2020,ALABAMA,849624.0,1441170.0,2290794.0,37.088625,62.911375,AL,-15.181210
1,2020,ALASKA,153778.0,189951.0,343729.0,44.738151,55.261849,AK,-7.531684
2,2020,ARIZONA,1672143.0,1661686.0,3333829.0,50.156832,49.843168,AZ,-2.113003
3,2020,ARKANSAS,423932.0,760647.0,1184579.0,35.787567,64.212433,AR,-16.482268
4,2020,CALIFORNIA,11110250.0,6006429.0,17116679.0,64.908911,35.091089,CA,12.639076
5,2020,COLORADO,1804352.0,1364607.0,3168959.0,56.938319,43.061681,CO,4.668485
6,2020,CONNECTICUT,1080831.0,714717.0,1795548.0,60.195049,39.804951,CT,7.925214
7,2020,DELAWARE,296268.0,200603.0,496871.0,59.626744,40.373256,DE,7.356909
9,2020,FLORIDA,5297045.0,5668731.0,10965776.0,48.305245,51.694755,FL,-3.964590
10,2020,GEORGIA,2473633.0,2461854.0,4935487.0,50.119330,49.880670,GA,-2.150505


In [114]:
def calc_pvi(state_po):
    df = pres_pivot[pres_pivot['state_po'] == state_po].sort_values(by=['year'], ascending=True)
    lean_20 = df[df['year'] == 2020]['lean'].values[0]
    lean_24 = df[df['year'] == 2024]['lean'].values[0]
    pvi = 0.25 * lean_20 + 0.75 * lean_24
    return pvi

def calc_dem_prev(state_po):
    df = pres_pivot[pres_pivot['state_po'] == state_po].sort_values(by=['year'], ascending=True)
    dem_24 = df[df['year'] == 2024]['dem_2p_pct'].values[0]
    return dem_24

In [115]:
candinfo['pvi'] = candinfo['state_po'].map(calc_pvi)
candinfo['dem_2p_24'] = candinfo['state_po'].map(calc_dem_prev)
candinfo.head(3)

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007


In [116]:
demo = pd.read_csv('../../2026_data/senate_demographics.csv')
demo = demo.drop(['Unnamed: 0'], axis=1)
demo.head()

,state_po,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,state,urban,rural,total,urban_pct,rural_pct
0,AL,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,Alabama,2900880,2123399,5024279,57.737240,42.262760
1,AK,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,Alaska,475967,257424,733391,64.899487,35.100513
2,AZ,61.913244,2.926293,24.346069,4.451431,3.328670,33.335593,Arizona,6385230,766272,7151502,89.285160,10.714840
3,AR,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,Arkansas,1670677,1340847,3011524,55.476131,44.523869
4,CA,42.658592,15.153478,32.366808,6.278722,0.317907,37.082883,California,37259490,2278733,39538223,94.236633,5.763367


In [117]:
candinfo = pd.merge(left=candinfo, right=demo.drop(['state'], axis=1), on='state_po', how='left')
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,...,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,...,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883


In [118]:
pol_df = pd.read_csv('../house/transformed/2026_house_prediction_dataset.csv')
polariz = pol_df['polarization'].values[0]
candinfo['polarization'] = np.full(candinfo.shape[0], polariz)

In [119]:
genbal = pd.read_csv('../../../snoutcounter-backend/averages/generic_ballot.csv')
gb_curr = genbal['net'].values[-1]
gb_curr

np.float64(-8.620113276112855)

In [120]:
candinfo['generic_ballot_avg'] = np.full(shape=candinfo.shape[0], fill_value=gb_curr)
candinfo.head(3)

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.620113
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.620113
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.620113


In [121]:
candinfo['dem_inc_dummy'] = candinfo['dem_inc_any'].map(lambda x: 0 if x == False else 1)
candinfo['rep_inc_dummy'] = candinfo['rep_inc_any'].map(lambda x: 0 if x == False else 1)
candinfo.head(3)

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.620113,0,0
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.620113,0,1
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.620113,0,1


In [122]:
poll_avg = pd.read_csv('transformed/senate_polling_averages.csv')
poll_avg.head()

,state,candidate_name,party,avg,std,lower_ci,upper_ci,effn
0,AK,Dan Sullivan,REP,48.488694,0.635873,47.242382,49.735006,2.323437
1,AK,Mary Peltola,DEM,49.473113,2.919672,43.750556,55.195670,2.323437
2,AL,Barry Moore,REP,47.000000,0.000000,47.000000,47.000000,0.053754
3,AL,Everett Wess,DEM,32.000000,0.000000,32.000000,32.000000,0.053754
4,AR,Hallie Shoffner,DEM,38.189521,5.356364,27.691047,48.687994,0.494462


In [123]:
poll_avg['party'].value_counts()

party
REP    26
DEM    24
IND     3
Name: count, dtype: int64

In [124]:
def get_polling_average(state, party, candidate_name):
    df = poll_avg[
        (poll_avg['state'] == state) &
        (poll_avg['party'].isin([party, 'IND']))
    ]
    if df.shape[0] == 0 or process.extractOne(candidate_name, df['candidate_name'], scorer=fuzz.token_sort_ratio, score_cutoff=30) is None:
        # lax score cutoff for fuzzy string matching - make sure to check manually when appropriate
        return 0, 0
    avg = df['avg'].values[0]
    effn = df['effn'].values[0]
    return (avg, effn)

In [125]:
for party in ['dem', 'rep']:
    candinfo[[f'{party}_poll_avg', f'{party}_effn']] = candinfo.apply(lambda x: get_polling_average(x['state_po'],
                                                                                                   party.upper(), x[f'{party}_cand']), axis=1,
                                                                     result_type='expand')
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,57.737240,42.262760,0.987401,-8.620113,0,0,32.000000,0.053754,47.000000,0.053754
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,64.899487,35.100513,0.987401,-8.620113,0,1,49.473113,2.323437,48.488694,2.323437
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,55.476131,44.523869,0.987401,-8.620113,0,1,38.189521,0.494462,49.190029,0.494462
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,...,86.026707,13.973293,0.987401,-8.620113,1,0,0.000000,0.000000,0.000000,0.000000
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,...,82.612117,17.387883,0.987401,-8.620113,1,0,0.000000,0.000000,0.000000,0.000000


In [126]:
state_to_census_region = {
    'Connecticut': 'New England',
    'Maine': 'New England',
    'Massachusetts': 'New England',
    'New Hampshire': 'New England',
    'Rhode Island': 'New England',
    'Vermont': 'New England',
    'New Jersey': 'Mid-Atlantic',
    'New York': 'Mid-Atlantic',
    'Pennsylvania': 'Mid-Atlantic',
    'Illinois': 'East North Central',
    'Indiana': 'East North Central',
    'Michigan': 'East North Central',
    'Ohio': 'East North Central',
    'Wisconsin': 'East North Central',
    'Iowa': 'West North Central',
    'Kansas': 'West North Central',
    'Minnesota': 'West North Central',
    'Missouri': 'West North Central',
    'Nebraska': 'West North Central',
    'North Dakota': 'West North Central',
    'South Dakota': 'West North Central',
    'Delaware': 'South Atlantic',
    'Maryland': 'South Atlantic',
    'Florida': 'South Atlantic',
    'Georgia': 'South Atlantic',
    'North Carolina': 'South Atlantic',
    'South Carolina': 'South Atlantic',
    'Virginia': 'South Atlantic',
    'West Virginia': 'South Atlantic',
    'Alabama': 'East South Central',
    'Kentucky': 'East South Central',
    'Mississippi': 'East South Central',
    'Tennessee': 'East South Central',
    'Arkansas': 'West South Central',
    'Louisiana': 'West South Central',
    'Oklahoma': 'West South Central',
    'Texas': 'West South Central',
    'Arizona': 'Mountain',
    'Colorado': 'Mountain',
    'Idaho': 'Mountain',
    'Montana': 'Mountain',
    'Nevada': 'Mountain',
    'New Mexico': 'Mountain',
    'Utah': 'Mountain',
    'Wyoming': 'Mountain',
    'California': 'Pacific',
    'Alaska': 'Pacific',
    'Oregon': 'Pacific',
    'Washington': 'Pacific',
    'Hawaii': 'Pacific'
}

In [127]:
candinfo['census_region'] = candinfo['state'].map(state_to_census_region)

In [128]:
candinfo['year'] = np.full(candinfo.shape[0], 2026)

In [129]:
candinfo.head(3)

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,0.987401,-8.620113,0,0,32.000000,0.053754,47.000000,0.053754,East South Central,2026
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,0.987401,-8.620113,0,1,49.473113,2.323437,48.488694,2.323437,Pacific,2026
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,0.987401,-8.620113,0,1,38.189521,0.494462,49.190029,0.494462,West South Central,2026


In [130]:
## Political scandals
# List of scandals by federal and state officeholders courtesy of Nathaniel Rakich
scandals = pd.read_csv('https://docs.google.com/spreadsheets/d/1ksBLxRR3GCZd33IvhkcNqqBd5K8HwlWC7YuAkVmS1lg/export?format=csv')
oth_scandals = pd.read_csv('../../2026_data/other_scandals.csv')
scandals = scandals.set_axis(['politician', 'state', 'position', 'year', 'type', 'outcome', 'comeback'], axis=1)
scandals['politician'] = scandals['politician'].map(lambda x: unicodedata.normalize('NFC', x))
scandals = pd.concat([scandals, oth_scandals], axis=0)
scandals.head()

,politician,state,position,year,type,outcome,comeback
0,Dale Caldwell,New Jersey,Lieutenant Governor,2026,"Sexual harassment, ethics",Resigned,NaN
1,Jimmy Gomez,California,Representative,2026,Affair with staffer,TBD,NaN
2,Liz Murrill,Louisiana,Attorney General,2026,Intimidation,TBD,NaN
3,Jim Costa,California,Representative,2026,Sexual harassment,TBD,NaN
4,Chuck Edwards,North Carolina,Representative,2026,Sexual harassment,Retired,NaN


In [131]:
def get_scandals(politician, state, cycle):
    # :param cycle: The year of the election cycle
    mask = ((scandals['state'] == state) & (scandals['year'] <= cycle))
    scan_df = scandals[mask]
    fuzzymatch = process.extractOne(politician, scan_df['politician'], scorer=fuzz.WRatio, score_cutoff=90)
    if fuzzymatch is None:
        return 0
    scan_df_pol = scan_df[scan_df['politician'] == fuzzymatch[0]]
    # Exponential decay factor to put less weight on older scandals relative to newer scandals
    # e^(\delta_t/8)
    scan_df_pol['decay_factor'] = np.exp(-(scan_df_pol['year'].astype(int).map(lambda x: cycle - x))/8)
    pol_scandal_score = scan_df_pol.groupby(['politician']).agg({'decay_factor': 'sum'})['decay_factor'].values[0]
    return pol_scandal_score

In [132]:
candinfo['dem_scandal_score'] = candinfo.apply(lambda x: get_scandals(x['dem_cand'], x['state'], x['year']), axis=1)
candinfo['rep_scandal_score'] = candinfo.apply(lambda x: get_scandals(x['rep_cand'], x['state'], x['year']), axis=1)
candinfo['net_scandal_score'] = candinfo['dem_scandal_score'] - candinfo['rep_scandal_score']
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,0,32.000000,0.053754,47.000000,0.053754,East South Central,2026,0.000000,0.0,0.000000
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,1,49.473113,2.323437,48.488694,2.323437,Pacific,2026,0.000000,0.0,0.000000
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,1,38.189521,0.494462,49.190029,0.494462,West South Central,2026,0.000000,0.0,0.000000
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,...,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879
4,Delaware,Chris Coons,Michael Katz,True,False,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,...,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000


In [133]:
clust_model = joblib.load('../../model/demographic_kmeans.pkl')
candinfo['demo_cluster'] = clust_model.predict(candinfo[['cvap_white_pct', 'cvap_aapi_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_natam_pct', 'college',
                                                        'urban_pct']])

In [134]:
candinfo.head(3)

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,...,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster
0,Alabama,Everett Weiss,Barry Moore,False,False,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,...,32.000000,0.053754,47.000000,0.053754,East South Central,2026,0.0,0.0,0.0,5
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,...,49.473113,2.323437,48.488694,2.323437,Pacific,2026,0.0,0.0,0.0,5
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,...,38.189521,0.494462,49.190029,0.494462,West South Central,2026,0.0,0.0,0.0,5


In [135]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_poll_avg'] = candinfo[f'{party}_poll_avg'].fillna(0)

In [138]:
candinfo['chamber'] = np.full(candinfo.shape[0], "Senate")

In [137]:
candinfo.columns.values

array(['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any',
       'state_po', 'fec_dem_fuzzymatch', 'fec_rep_fuzzymatch',
       'dem_funds', 'rep_funds', '2p_funds', 'dem_funds_2p_pct',
       'rep_funds_2p_pct', 'pvi', 'dem_2p_24', 'cvap_white_pct',
       'cvap_aapi_pct', 'cvap_hisp_pct', 'cvap_black_pct',
       'cvap_natam_pct', 'college', 'urban', 'rural', 'total',
       'urban_pct', 'rural_pct', 'polarization', 'generic_ballot_avg',
       'dem_inc_dummy', 'rep_inc_dummy', 'dem_poll_avg', 'dem_effn',
       'rep_poll_avg', 'rep_effn', 'census_region', 'year',
       'dem_scandal_score', 'rep_scandal_score', 'net_scandal_score',
       'demo_cluster'], dtype=object)

In [139]:
candinfo.to_csv('transformed/2026_senate_prediction_dataset.csv')